# 01 — Exploration du pipeline DeepVox Phase 1

Ce notebook valide chaque étape du pipeline :
1. Chargement d'un échantillon Common Voice FR
2. Resampling 16 → 8 kHz
3. Encodage Codec2 1200 bps
4. Décodage et écoute de la reconstruction
5. Visualisation des features (bits, mel, PCM)
6. Test du classifieur BiLSTM sur données synthétiques

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import IPython.display as ipd
from pathlib import Path
from tqdm.auto import tqdm

from deepvox.codec2.encoder import (
    encode_pcm, decode_frames, unpack_frames, add_delta_features,
    wav_to_frames, SAMPLE_RATE, SAMPLES_PER_FRAME, FRAME_DURATION_MS
)
from deepvox.data.preprocess import FRENCH_PHONEMES, NUM_PHONEMES

print(f'Phonèmes français : {NUM_PHONEMES}')
print(FRENCH_PHONEMES)

## 1. Découverte des données Common Voice FR

In [ ]:
import pandas as pd

DATA_DIR = Path('../data')

# Chercher le fichier TSV
tsv_files = list(DATA_DIR.rglob('validated*.tsv')) + list(DATA_DIR.rglob('train*.tsv'))
print('Fichiers TSV trouvés :')
for f in tsv_files:
    print(f'  {f}')

if tsv_files:
    df = pd.read_csv(tsv_files[0], sep='\t', nrows=1000)
    print(f'\nColonnes : {list(df.columns)}')
    print(f'Lignes (sample) : {len(df)}')
    df.head()

In [ ]:
# Trouver un fichier audio d'exemple
clips_dirs = list(DATA_DIR.rglob('clips'))
if clips_dirs:
    clips_dir = clips_dirs[0]
    sample_files = list(clips_dir.glob('*.mp3'))[:5]
    print(f'Clips dir : {clips_dir}')
    print(f'Nombre total de clips : {len(list(clips_dir.glob("*.mp3")))}')
    print(f'Exemples : {[f.name for f in sample_files]}')
else:
    print('Dossier clips non trouvé — vérifier le téléchargement')

## 2. Pipeline audio : original → resample → Codec2 → reconstruction

In [ ]:
# Charger un fichier audio d'exemple
if sample_files:
    sample_path = sample_files[0]
    print(f'Fichier : {sample_path.name}')
    
    # Audio original (16 kHz)
    audio_16k, sr_16k = librosa.load(sample_path, sr=16000, mono=True)
    print(f'Original : {len(audio_16k)} samples @ {sr_16k} Hz = {len(audio_16k)/sr_16k:.2f}s')
    
    # Resample à 8 kHz
    audio_8k = librosa.resample(audio_16k, orig_sr=sr_16k, target_sr=8000)
    print(f'Resampled : {len(audio_8k)} samples @ 8000 Hz = {len(audio_8k)/8000:.2f}s')
    
    # Écouter l'original
    print('\n▶ Audio original (16 kHz) :')
    ipd.display(ipd.Audio(audio_16k, rate=sr_16k))
    
    print('\n▶ Audio resampled (8 kHz) :')
    ipd.display(ipd.Audio(audio_8k, rate=8000))

In [ ]:
# Encodage Codec2
if sample_files:
    pcm = (audio_8k * 32767).astype(np.int16)
    frames = encode_pcm(pcm)
    print(f'Codec2 frames : {frames.shape} (n_frames, bytes_per_frame)')
    print(f'Durée couverte : {len(frames) * FRAME_DURATION_MS / 1000:.2f}s')
    print(f'Taille compressée : {frames.nbytes} bytes vs PCM : {pcm.nbytes} bytes')
    print(f'Ratio de compression : {pcm.nbytes / frames.nbytes:.1f}x')
    
    # Décodage
    pcm_reconstructed = decode_frames(frames)
    audio_reconstructed = pcm_reconstructed.astype(np.float32) / 32767.0
    print(f'\nReconstructed : {len(pcm_reconstructed)} samples')
    
    print('\n▶ Audio reconstruit depuis Codec2 :')
    ipd.display(ipd.Audio(audio_reconstructed, rate=8000))

## 3. Visualisation des représentations

In [ ]:
if sample_files:
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), constrained_layout=True)
    
    # 1. Forme d'onde
    time_axis = np.arange(len(audio_8k)) / 8000
    axes[0].plot(time_axis, audio_8k, linewidth=0.5)
    axes[0].set_title('Forme d\'onde (8 kHz)')
    axes[0].set_xlabel('Temps (s)')
    axes[0].set_ylabel('Amplitude')
    
    # 2. Spectrogramme mel
    mel = librosa.feature.melspectrogram(y=audio_8k, sr=8000, n_mels=80, n_fft=1024)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    librosa.display.specshow(mel_db, sr=8000, x_axis='time', y_axis='mel', ax=axes[1])
    axes[1].set_title('Spectrogramme Mel (80 bandes) — Condition C')
    
    # 3. Features Codec2 (48 bits)
    feats = unpack_frames(frames)
    axes[2].imshow(feats.T, aspect='auto', cmap='binary', interpolation='nearest')
    axes[2].set_title(f'Codec2 raw features (48 bits/frame) — Condition A — {feats.shape[0]} frames')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Bit index')
    
    # 4. Features Codec2 + delta (96)
    feats_delta = add_delta_features(feats)
    axes[3].imshow(feats_delta.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    axes[3].set_title(f'Codec2 + delta features (96/frame) — Condition B')
    axes[3].set_xlabel('Frame')
    axes[3].set_ylabel('Feature index')
    
    plt.savefig('../outputs/exploration_features.png', dpi=150)
    plt.show()
    print('Figure sauvegardée → outputs/exploration_features.png')

## 4. Comparaison qualitative : original vs Codec2

In [ ]:
if sample_files:
    # SNR entre original et reconstruit
    min_len = min(len(audio_8k), len(audio_reconstructed))
    orig = audio_8k[:min_len]
    recon = audio_reconstructed[:min_len]
    
    noise = orig - recon
    snr = 10 * np.log10(np.sum(orig**2) / (np.sum(noise**2) + 1e-10))
    print(f'SNR (original vs Codec2 reconstruit) : {snr:.1f} dB')
    
    # Corrélation
    corr = np.corrcoef(orig, recon)[0, 1]
    print(f'Corrélation : {corr:.4f}')
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), constrained_layout=True)
    t = np.arange(min_len) / 8000
    axes[0].plot(t, orig, linewidth=0.5, label='Original')
    axes[0].plot(t, recon, linewidth=0.5, alpha=0.7, label='Codec2')
    axes[0].set_title('Comparaison temporelle')
    axes[0].legend()
    axes[0].set_xlabel('Temps (s)')
    
    axes[1].plot(t, noise, linewidth=0.5, color='red')
    axes[1].set_title(f'Erreur de reconstruction (SNR = {snr:.1f} dB)')
    axes[1].set_xlabel('Temps (s)')
    
    plt.show()

## 5. Test du classifieur sur données synthétiques

In [ ]:
import torch
from deepvox.models.phoneme_classifier import PhonemeClassifier

# Tester les 4 conditions
for name, dim in [('A — Codec2 raw', 48), ('B — Codec2+delta', 96),
                   ('C — Mel spec', 80), ('D — PCM raw', 320)]:
    model = PhonemeClassifier(input_dim=dim)
    x = torch.randn(4, 25, dim)  # batch=4, seq=25 frames (1s)
    out = model(x)
    print(f'{name:20s} | input={str(x.shape):20s} | output={str(out.shape):22s} | params={model.count_parameters():,}')

In [ ]:
# Encoder un batch de fichiers pour démontrer tqdm
if sample_files and len(list(clips_dir.glob('*.mp3'))) > 20:
    batch = list(clips_dir.glob('*.mp3'))[:50]

    stats = {
        'n_frames': [],
        'duration_s': [],
        'compression_ratio': [],
    }

    for mp3_path in tqdm(batch, desc='Encodage Codec2', unit='file'):
        audio, _ = librosa.load(mp3_path, sr=8000, mono=True)
        pcm = (audio * 32767).astype(np.int16)
        frames = encode_pcm(pcm)
        stats['n_frames'].append(len(frames))
        stats['duration_s'].append(len(pcm) / 8000)
        stats['compression_ratio'].append(pcm.nbytes / frames.nbytes)

    import pandas as pd
    df_stats = pd.DataFrame(stats)
    print(df_stats.describe())

    fig, axes = plt.subplots(1, 3, figsize=(14, 3), constrained_layout=True)
    axes[0].hist(stats['duration_s'], bins=20, edgecolor='black')
    axes[0].set_title('Durée (s)')
    axes[1].hist(stats['n_frames'], bins=20, edgecolor='black')
    axes[1].set_title('Nombre de frames Codec2')
    axes[2].hist(stats['compression_ratio'], bins=20, edgecolor='black')
    axes[2].set_title('Ratio de compression PCM→Codec2')
    plt.show()

## 5b. Encodage batch avec tqdm

Démonstration : encoder plusieurs fichiers d'un coup, avec barre de progression.
`tqdm.auto` détecte automatiquement qu'on est dans un notebook → widget HTML interactif.

## 6. Statistiques du dataset

In [ ]:
if tsv_files:
    df_full = pd.read_csv(tsv_files[0], sep='\t')
    print(f'Nombre total d\'énoncés : {len(df_full):,}')
    
    if 'up_votes' in df_full.columns and 'down_votes' in df_full.columns:
        quality = df_full['up_votes'] - df_full['down_votes']
        print(f'Score qualité moyen : {quality.mean():.2f}')
    
    if 'locale' in df_full.columns:
        print(f'\nLocales :')
        print(df_full['locale'].value_counts().head(10))
    
    if 'sentence' in df_full.columns:
        lengths = df_full['sentence'].str.len()
        print(f'\nLongueur des phrases :')
        print(f'  min={lengths.min()}, max={lengths.max()}, mean={lengths.mean():.0f}')
        
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.hist(lengths, bins=50, edgecolor='black')
        ax.set_title('Distribution des longueurs de phrases')
        ax.set_xlabel('Nombre de caractères')
        ax.set_ylabel('Fréquence')
        plt.show()

## 7. Structure des champs Codec2 1200 bps

Chaque frame = 48 bits = 40 ms :
- **LSP** (Line Spectral Pairs) : bits 0-35 → 36 bits → envelope spectrale
- **Pitch** : bits 36-42 → 7 bits → fréquence fondamentale
- **Energy** : bits 43-47 → 5 bits → énergie du frame

In [ ]:
if sample_files:
    # Analyser la structure des champs
    lsp = feats[:, :36]
    pitch = feats[:, 36:43]
    energy = feats[:, 43:48]
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), constrained_layout=True)
    
    axes[0].imshow(lsp.T, aspect='auto', cmap='binary', interpolation='nearest')
    axes[0].set_title('LSP (36 bits) — Envelope spectrale')
    axes[0].set_ylabel('Bit')
    
    # Convertir pitch en valeur décimale
    pitch_vals = np.packbits(np.pad(pitch, ((0,0),(1,0)), constant_values=0), axis=1).astype(float).flatten()
    axes[1].plot(pitch_vals)
    axes[1].set_title('Pitch (7 bits) — Fréquence fondamentale')
    axes[1].set_ylabel('Valeur')
    
    energy_vals = np.packbits(np.pad(energy, ((0,0),(3,0)), constant_values=0), axis=1).astype(float).flatten()
    axes[2].plot(energy_vals)
    axes[2].set_title('Energy (5 bits)')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Valeur')
    
    plt.savefig('../outputs/codec2_fields.png', dpi=150)
    plt.show()
    print('Figure sauvegardée → outputs/codec2_fields.png')

In [ ]:
print('\n✅ Notebook d\'exploration terminé.')
print('Prochaine étape : lancer MFA sur le corpus pour obtenir les alignements phonétiques.')